# Lab

Build a stable invoice-report sort. Use Run; sign in before saving work. Each task includes its own data. Carry methods forward explicitly, rather than relying on state from another cell.

## 1. Recursively Total a Batch

```{index} 1. Recursively Total a Batch
```

Write a recursive suffix sum returning long. Empty input returns zero. State the helper promise and decreasing measure. Compare with a loop or library reference. Use small batches because suffix recursion has linear depth.

In [ ]:
using System;
using System.Linq;

int[] amounts={200,100,200,100,300};
// Your code starts here.
/* TODO: complete the task. */
// Your code ends here.

In [1]:
using System;
using System.Linq;

// Solution
int[] amounts={200,100,200,100,300};
Console.WriteLine($"total={Sum(amounts)}, empty={Sum(Array.Empty<int>())}");
if(Sum(amounts)!=amounts.Sum(x=>(long)x))throw new Exception("Sum mismatch.");
long Sum(int[] values)
{
 ArgumentNullException.ThrowIfNull(values);
 return From(0);
 // From(i) returns sum of [i,n); measure n-i.
 long From(int i)=>i==values.Length?0:values[i]+From(i+1);
}

total=900, empty=0


## 2. Merge Sorted Batches

```{index} 2. Merge Sorted Batches
```

Implement left-first merge of two sorted integer batches into a fresh array. Include tail copies; check exact output and empty-side behavior. State k=i+j and the consumed-prefix invariant.

In [ ]:
using System;
using System.Linq;

int[] left={100,200,200},right={100,300};
// Your code starts here.
/* TODO: complete the task. */
// Your code ends here.

In [1]:
using System;
using System.Linq;

// Solution
int[] left={100,200,200},right={100,300};
int[] result=Merge(left,right);
Console.WriteLine(string.Join(", ",result));
if(!result.SequenceEqual(new[]{100,100,200,200,300}))throw new Exception("Merge mismatch.");
Console.WriteLine($"empty-side length={Merge(left,Array.Empty<int>()).Length}");
int[] Merge(int[] a,int[] b)
{
 int[] result=new int[a.Length+b.Length];int i=0,j=0,k=0;
 while(i<a.Length&&j<b.Length)result[k++]=a[i]<=b[j]?a[i++]:b[j++];
 while(i<a.Length)result[k++]=a[i++];
 while(j<b.Length)result[k++]=b[j++];
 return result;
}

100, 100, 200, 200, 300
empty-side length=3


## 3. Complete the Recursive Sort

```{index} 3. Complete the Recursive Sort
```

Implement the range/buffer sort from the worked section, reusing the merge reasoning from Task 2. Return a fresh array and leave input unchanged. Include odd-length ranges. Print key comparisons and peak active frames.

In [ ]:
using System;
using System.Linq;

int[] amounts={200,100,200,100,300};
// Your code starts here.
/* TODO: complete the task. */
// Your code ends here.

In [1]:
using System;
using System.Linq;

// Solution
int[] amounts={200,100,200,100,300};
var counts=new StableSort.Counts();
int[] sorted=StableSort.Sort(amounts,(a,b)=>a.CompareTo(b),counts);
Console.WriteLine($"input={string.Join(",",amounts)}");
Console.WriteLine($"sorted={string.Join(",",sorted)}, comparisons={counts.Comparisons}, peak={counts.PeakFrames}");
public static class StableSort
{
    public sealed class Counts
    {
        public long Comparisons, Writes, Calls;
        public int PeakFrames;
    }
    // Inputs must already be sorted under compare. Equal left items win ties.
    public static T[] Merge<T>(T[] left, T[] right, Comparison<T> compare)
    {
        ArgumentNullException.ThrowIfNull(left);
        ArgumentNullException.ThrowIfNull(right);
        ArgumentNullException.ThrowIfNull(compare);
        T[] result = new T[checked(left.Length + right.Length)];
        int i=0,j=0,k=0;
        while(i<left.Length && j<right.Length)
            result[k++]=compare(left[i],right[j])<=0?left[i++]:right[j++];
        while(i<left.Length)result[k++]=left[i++];
        while(j<right.Length)result[k++]=right[j++];
        return result;
    }
    // Fresh array even for empty/singleton input. Source order is unchanged.
    public static T[] Sort<T>(T[] input, Comparison<T> compare, Counts counts)
    {
        ArgumentNullException.ThrowIfNull(input);
        ArgumentNullException.ThrowIfNull(compare);
        ArgumentNullException.ThrowIfNull(counts);
        counts.Comparisons=counts.Writes=counts.Calls=0;counts.PeakFrames=0;
        T[] values=(T[])input.Clone();
        T[] buffer=new T[values.Length];
        SortRange(0,values.Length,1);
        return values;

        void SortRange(int start,int end,int depth)
        {
            counts.Calls++;counts.PeakFrames=Math.Max(counts.PeakFrames,depth);
            if(end-start<=1)return;
            int middle=start+(end-start)/2;
            SortRange(start,middle,depth+1);
            SortRange(middle,end,depth+1);
            int i=start,j=middle,k=start;
            while(i<middle && j<end)
            {
                counts.Comparisons++;
                buffer[k++]=compare(values[i],values[j])<=0?values[i++]:values[j++];
                counts.Writes++;
            }
            while(i<middle){buffer[k++]=values[i++];counts.Writes++;}
            while(j<end){buffer[k++]=values[j++];counts.Writes++;}
            for(int p=start;p<end;p++){values[p]=buffer[p];counts.Writes++;}
        }
    }
}


input=200,100,200,100,300
sorted=100,100,200,200,300, comparisons=7, peak=4


## 4. Preserve Invoice Identity

```{index} 4. Preserve Invoice Identity
```

Extend Task 3 to labeled invoices. Compare amounts only. Verify that IDs B/D and A/C retain arrival order within their tie groups. Explain why left-first merging establishes stability.

In [ ]:
using System;
using System.Linq;

var invoices=new[]{("A",200),("B",100),("C",200),("D",100),("E",300)};
// Your code starts here.
/* TODO: complete the task. */
// Your code ends here.

In [1]:
using System;
using System.Linq;

// Solution
var invoices=new[]{("A",200),("B",100),("C",200),("D",100),("E",300)};
var sorted=StableSort.Sort(invoices,(a,b)=>a.Item2.CompareTo(b.Item2),new StableSort.Counts());
string[] ids=sorted.Select(x=>x.Item1).ToArray();
if(!ids.SequenceEqual(new[]{"B","D","A","C","E"}))throw new Exception("Stability mismatch.");
Console.WriteLine(string.Join(", ",ids));
public static class StableSort
{
    public sealed class Counts
    {
        public long Comparisons, Writes, Calls;
        public int PeakFrames;
    }
    // Inputs must already be sorted under compare. Equal left items win ties.
    public static T[] Merge<T>(T[] left, T[] right, Comparison<T> compare)
    {
        ArgumentNullException.ThrowIfNull(left);
        ArgumentNullException.ThrowIfNull(right);
        ArgumentNullException.ThrowIfNull(compare);
        T[] result = new T[checked(left.Length + right.Length)];
        int i=0,j=0,k=0;
        while(i<left.Length && j<right.Length)
            result[k++]=compare(left[i],right[j])<=0?left[i++]:right[j++];
        while(i<left.Length)result[k++]=left[i++];
        while(j<right.Length)result[k++]=right[j++];
        return result;
    }
    // Fresh array even for empty/singleton input. Source order is unchanged.
    public static T[] Sort<T>(T[] input, Comparison<T> compare, Counts counts)
    {
        ArgumentNullException.ThrowIfNull(input);
        ArgumentNullException.ThrowIfNull(compare);
        ArgumentNullException.ThrowIfNull(counts);
        counts.Comparisons=counts.Writes=counts.Calls=0;counts.PeakFrames=0;
        T[] values=(T[])input.Clone();
        T[] buffer=new T[values.Length];
        SortRange(0,values.Length,1);
        return values;

        void SortRange(int start,int end,int depth)
        {
            counts.Calls++;counts.PeakFrames=Math.Max(counts.PeakFrames,depth);
            if(end-start<=1)return;
            int middle=start+(end-start)/2;
            SortRange(start,middle,depth+1);
            SortRange(middle,end,depth+1);
            int i=start,j=middle,k=start;
            while(i<middle && j<end)
            {
                counts.Comparisons++;
                buffer[k++]=compare(values[i],values[j])<=0?values[i++]:values[j++];
                counts.Writes++;
            }
            while(i<middle){buffer[k++]=values[i++];counts.Writes++;}
            while(j<end){buffer[k++]=values[j++];counts.Writes++;}
            for(int p=start;p<end;p++){values[p]=buffer[p];counts.Writes++;}
        }
    }
}


B, D, A, C, E


## 5. Verify the Report Contract

```{index} 5. Verify the Report Contract
```

Use your complete sort to verify order, occurrence preservation, unchanged input, and fresh-array identity. Test empty, singleton, duplicate, odd, reverse, and extreme inputs. Compare with an independent reference; sortedness alone is insufficient.

In [ ]:
using System;
using System.Linq;

int[][] cases={Array.Empty<int>(),new[]{200},new[]{200,100,200},new[]{300,200,100},new[]{int.MaxValue,int.MinValue}};
// Your code starts here.
/* TODO: complete the task. */
// Your code ends here.

In [1]:
using System;
using System.Linq;

// Solution
int[][] cases={Array.Empty<int>(),new[]{200},new[]{200,100,200},new[]{300,200,100},new[]{int.MaxValue,int.MinValue}};
foreach(int[] values in cases)
{
 int[] before=(int[])values.Clone();
 int[] actual=StableSort.Sort(values,(a,b)=>a.CompareTo(b),new StableSort.Counts());
 if(!actual.SequenceEqual(before.OrderBy(x=>x))||!values.SequenceEqual(before)||ReferenceEquals(values,actual))throw new Exception("Sort contract failed.");
 Console.WriteLine($"n={values.Length}: order, preservation, input, and fresh copy passed");
}
public static class StableSort
{
    public sealed class Counts
    {
        public long Comparisons, Writes, Calls;
        public int PeakFrames;
    }
    // Inputs must already be sorted under compare. Equal left items win ties.
    public static T[] Merge<T>(T[] left, T[] right, Comparison<T> compare)
    {
        ArgumentNullException.ThrowIfNull(left);
        ArgumentNullException.ThrowIfNull(right);
        ArgumentNullException.ThrowIfNull(compare);
        T[] result = new T[checked(left.Length + right.Length)];
        int i=0,j=0,k=0;
        while(i<left.Length && j<right.Length)
            result[k++]=compare(left[i],right[j])<=0?left[i++]:right[j++];
        while(i<left.Length)result[k++]=left[i++];
        while(j<right.Length)result[k++]=right[j++];
        return result;
    }
    // Fresh array even for empty/singleton input. Source order is unchanged.
    public static T[] Sort<T>(T[] input, Comparison<T> compare, Counts counts)
    {
        ArgumentNullException.ThrowIfNull(input);
        ArgumentNullException.ThrowIfNull(compare);
        ArgumentNullException.ThrowIfNull(counts);
        counts.Comparisons=counts.Writes=counts.Calls=0;counts.PeakFrames=0;
        T[] values=(T[])input.Clone();
        T[] buffer=new T[values.Length];
        SortRange(0,values.Length,1);
        return values;

        void SortRange(int start,int end,int depth)
        {
            counts.Calls++;counts.PeakFrames=Math.Max(counts.PeakFrames,depth);
            if(end-start<=1)return;
            int middle=start+(end-start)/2;
            SortRange(start,middle,depth+1);
            SortRange(middle,end,depth+1);
            int i=start,j=middle,k=start;
            while(i<middle && j<end)
            {
                counts.Comparisons++;
                buffer[k++]=compare(values[i],values[j])<=0?values[i++]:values[j++];
                counts.Writes++;
            }
            while(i<middle){buffer[k++]=values[i++];counts.Writes++;}
            while(j<end){buffer[k++]=values[j++];counts.Writes++;}
            for(int p=start;p<end;p++){values[p]=buffer[p];counts.Writes++;}
        }
    }
}


n=0: order, preservation, input, and fresh copy passed
n=1: order, preservation, input, and fresh copy passed
n=3: order, preservation, input, and fresh copy passed
n=3: order, preservation, input, and fresh copy passed
n=2: order, preservation, input, and fresh copy passed


## Project Connection

```{index} Project Connection
```

Identify a project report where equal-key record order matters. Specify its comparison key, tie policy, input-preservation policy, and expected time/auxiliary space. If recursion does not suit the project, explain the iterative alternative.

## Submit

```{index} Submit
```

Submit code, verified outputs, and the requested contracts/invariants through the assignment panel when available.